# Reconstruct minimal state for Tableau exports

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reload the saved model

In [6]:
import tensorflow as tf

models_path = '/content/drive/MyDrive/June/Final Project - Retinal Screening Triage System/models/'
model_ft = tf.keras.models.load_model(models_path + 'efficientnetb0_finetuned.keras')

Reconstruct data split and pipeline (same as Blocks 5/8)

In [7]:
import pandas as pd
import os

drive_path = '/content/drive/MyDrive/June/Final Project - Retinal Screening Triage System/data/raw/'
train_csv = pd.read_csv(os.path.join(drive_path, 'train.csv'))

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    train_csv, test_size=0.30, stratify=train_csv['diagnosis'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['diagnosis'], random_state=42
)

print(f"Val: {len(val_df)}")

Val: 549


Download images directly via kagglehub (faster than Drive, as established)

In [9]:
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

import kagglehub
path = kagglehub.competition_download('aptos2019-blindness-detection')
local_path = os.path.join(path, 'train_images') + '/'

100%|██████████| 9.51G/9.51G [07:26<00:00, 22.9MB/s]

Extracting files...


In [10]:
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = 224
BATCH_SIZE = 32

def load_and_preprocess_image(id_code, label):
    img_path = tf.strings.join([local_path, id_code, '.png'])
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = preprocess_input(img)
    return img, label

def build_dataset(df, shuffle=False):
    ids = df['id_code'].values
    labels = df['diagnosis'].values
    ds = tf.data.Dataset.from_tensor_slices((ids, labels))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

val_ds = build_dataset(val_df)
print("val_ds built")

val_ds built


Get predictions and true labels

In [11]:
import numpy as np

y_val_true = []
y_val_probs = []

for images, labels in val_ds:
    preds = model_ft.predict(images, verbose=0)
    y_val_probs.extend(preds)
    y_val_true.extend(labels.numpy())

y_val_true = np.array(y_val_true)
y_val_probs = np.array(y_val_probs)
y_val_pred = np.argmax(y_val_probs, axis=1)

Export confusion matrix as CSV

In [12]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_val_true, y_val_pred)
cm_df = pd.DataFrame(cm, index=[f'True_{i}' for i in range(5)], columns=[f'Pred_{i}' for i in range(5)])

processed_path = '/content/drive/MyDrive/June/Final Project - Retinal Screening Triage System/data/processed/'
cm_df.to_csv(processed_path + 'confusion_matrix.csv')
print(cm_df)

        Pred_0  Pred_1  Pred_2  Pred_3  Pred_4
True_0     258       4       5       1       3
True_1      13      17      16       4       5
True_2      10       5      53      58      24
True_3       0       0       2      20       7
True_4       0       3       4      17      20


Export trade-off curve data

In [13]:
y_val_confidence = np.max(y_val_probs, axis=1)
is_severe = np.isin(y_val_true, [3, 4])

thresholds = np.arange(0.5, 1.0, 0.05)
rows = []
for t in thresholds:
    automated_mask = y_val_confidence >= t
    automation_rate = automated_mask.mean()
    severe_automated_mask = automated_mask & is_severe
    severe_recall = (y_val_pred[severe_automated_mask] == y_val_true[severe_automated_mask]).mean() if severe_automated_mask.sum() > 0 else None
    rows.append({'threshold': t, 'automation_rate': automation_rate, 'severe_recall': severe_recall})

tradeoff_df = pd.DataFrame(rows)
tradeoff_df.to_csv(processed_path + 'tradeoff_curve.csv', index=False)
print(tradeoff_df)

   threshold  automation_rate  severe_recall
0       0.50         0.670310       0.684211
1       0.55         0.617486       0.696970
2       0.60         0.570128       0.800000
3       0.65         0.522769       0.736842
4       0.70         0.491803       0.642857
5       0.75         0.468124       0.700000
6       0.80         0.442623       0.750000
7       0.85         0.408015       0.600000
8       0.90         0.358834       1.000000
9       0.95         0.269581            NaN
